## Step 1: load bp examination data into database

In [38]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, types # Export to DataBase

df_exam_bp = pd.read_sas("../data/examination_data/BPXO_L.xpt", format="xport", encoding="utf-8")

df_exam_bp.head()

,SEQN,BPAOARM,BPAOCSZ,BPXOSY1,BPXODI1,BPXOSY2,BPXODI2,BPXOSY3,BPXODI3,BPXOPLS1,BPXOPLS2,BPXOPLS3
0,130378.0,R,4.0,135.0,98.0,131.0,96.0,132.0,94.0,82.0,79.0,82.0
1,130379.0,R,4.0,121.0,84.0,117.0,76.0,113.0,76.0,72.0,71.0,73.0
2,130380.0,R,4.0,111.0,79.0,112.0,80.0,104.0,76.0,84.0,83.0,77.0
3,130386.0,R,4.0,110.0,72.0,120.0,74.0,115.0,75.0,59.0,64.0,64.0
4,130387.0,R,4.0,143.0,76.0,136.0,74.0,145.0,78.0,80.0,80.0,77.0


In [39]:
print("Blood Pressure Examination Data Info:")
df_exam_bp.info() 

Blood Pressure Examination Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7801 entries, 0 to 7800
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      7801 non-null   float64
 1   BPAOARM   7801 non-null   object 
 2   BPAOCSZ   7611 non-null   float64
 3   BPXOSY1   7517 non-null   float64
 4   BPXODI1   7517 non-null   float64
 5   BPXOSY2   7505 non-null   float64
 6   BPXODI2   7505 non-null   float64
 7   BPXOSY3   7480 non-null   float64
 8   BPXODI3   7480 non-null   float64
 9   BPXOPLS1  7517 non-null   float64
 10  BPXOPLS2  7505 non-null   float64
 11  BPXOPLS3  7480 non-null   float64
dtypes: float64(11), object(1)
memory usage: 731.5+ KB


In [40]:
df_exam_bp.columns

Index(['SEQN', 'BPAOARM', 'BPAOCSZ', 'BPXOSY1', 'BPXODI1', 'BPXOSY2',
       'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3'],
      dtype='object')

In [41]:
# Select and rename essential columns
    # We'll keep SEQN for merging
    # We'll select systolic Blood Pressure, Diastolic Blood Pressure, and Pulse data.

blood_pressure_columns = {
    'SEQN': 'Participant_ID',
    
    # Systolic Blood Pressure (收缩压)
    'BPXOSY1': 'Systolic_BP_1',
    'BPXOSY2': 'Systolic_BP_2',
    'BPXOSY3': 'Systolic_BP_3',

    # Diastolic Blood Pressure (舒张压)
    'BPXODI1': 'Diastolic_BP_1',
    'BPXODI2': 'Diastolic_BP_2',
    'BPXODI3': 'Diastolic_BP_3',

    # Pulse
    'BPXOPLS1': 'Pulse_1',
    'BPXOPLS2': 'Pulse_2',
    'BPXOPLS3': 'Pulse_3'
}


df_exam_bp_selected = df_exam_bp[list(blood_pressure_columns.keys())].copy() # list(...) transfers dicts into lists, which then can be worked in dataframe. 
df_exam_bp_selected.rename(columns=blood_pressure_columns, inplace=True) 

print("--- Selected and Renamed Blood Pressure Examination Data Info ---")
df_exam_bp_selected.info()

--- Selected and Renamed Blood Pressure Examination Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7801 entries, 0 to 7800
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Participant_ID  7801 non-null   float64
 1   Systolic_BP_1   7517 non-null   float64
 2   Systolic_BP_2   7505 non-null   float64
 3   Systolic_BP_3   7480 non-null   float64
 4   Diastolic_BP_1  7517 non-null   float64
 5   Diastolic_BP_2  7505 non-null   float64
 6   Diastolic_BP_3  7480 non-null   float64
 7   Pulse_1         7517 non-null   float64
 8   Pulse_2         7505 non-null   float64
 9   Pulse_3         7480 non-null   float64
dtypes: float64(10)
memory usage: 609.6 KB


In [42]:
# Calculate hypertension
df_exam_bp_selected['Systolic_BP_Avg'] = df_exam_bp_selected[['Systolic_BP_1', 'Systolic_BP_2', 'Systolic_BP_3']].mean(axis=1).round() # axis=1 correctly calculates the mean across the columns (horizontally) for each row (participant).
df_exam_bp_selected['Diastolic_BP_Avg'] = df_exam_bp_selected[['Diastolic_BP_1', 'Diastolic_BP_2', 'Diastolic_BP_3']].mean(axis=1).round()
df_exam_bp_selected.convert_dtypes() 

,Participant_ID,Systolic_BP_1,Systolic_BP_2,Systolic_BP_3,Diastolic_BP_1,Diastolic_BP_2,Diastolic_BP_3,Pulse_1,Pulse_2,Pulse_3,Systolic_BP_Avg,Diastolic_BP_Avg
0,130378,135,131,132,98,96,94,82,79,82,133,96
1,130379,121,117,113,84,76,76,72,71,73,117,79
2,130380,111,112,104,79,80,76,84,83,77,109,78
3,130386,110,120,115,72,74,75,59,64,64,115,74
4,130387,143,136,145,76,74,78,80,80,77,141,76
...,...,...,...,...,...,...,...,...,...,...,...,...
7796,142306,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7797,142307,127,132,131,75,73,72,71,70,67,130,73
7798,142308,106,106,112,65,69,74,58,61,69,108,69
7799,142309,127,125,128,81,82,81,80,79,83,127,81


In [43]:
# export as csv
file_path = "../data/examination_data/cleaned_blood_pressure_examination_data.csv" 

try:
    df_exam_bp_selected.to_csv(file_path, index=False, encoding='utf-8')
    print(f"DataFrame successfully saved to: {file_path}")
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")

DataFrame successfully saved to: ../data/examination_data/cleaned_blood_pressure_examination_data.csv


In [45]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 

In [46]:
df_exam_bp_selected.to_sql(name = 'blood_pressure_exmaination_data', con=engine, schema='capstone_group_3',if_exists='replace',index=False)

801

## Step 2: combine bp examination data with questionaire data and load into database

In [47]:
# read blood pressure questionaire data
df_qn = pd.read_csv("../data/questionaire_data/cleaned_blood_pressure_questionaire_data.csv")
df_qn

,Participant_ID,Ever_Had_Hypertension,Hypertension_Told_Twice,Currently_Taking_BP_Meds,Ever_Had_High_Cholesterol,Currently_Taking_Lower_Blood_Cholesterol_Meds
0,130378,1.0,1.0,1.0,2.0,2.0
1,130379,1.0,1.0,1.0,2.0,2.0
2,130380,2.0,NaN,NaN,1.0,1.0
3,130384,2.0,NaN,NaN,2.0,2.0
4,130385,2.0,NaN,NaN,2.0,2.0
...,...,...,...,...,...,...
8496,142305,1.0,1.0,1.0,1.0,1.0
8497,142307,2.0,NaN,NaN,1.0,1.0
8498,142308,2.0,NaN,NaN,2.0,2.0
8499,142309,2.0,NaN,NaN,2.0,2.0


In [51]:
print(df_qn.shape)
print(df_exam_bp_selected.shape)

(8501, 6)
(7801, 12)


In [53]:
print(df_qn['Participant_ID'].duplicated().sum())
print(df_exam_bp_selected['Participant_ID'].duplicated().sum())


0
0


In [54]:
df_bp_merged = df_exam_bp_selected.merge(
    df_qn[['Participant_ID', 'Currently_Taking_BP_Meds']],
    on='Participant_ID',
    how='left'
)
df_bp_merged

,Participant_ID,Systolic_BP_1,Systolic_BP_2,Systolic_BP_3,Diastolic_BP_1,Diastolic_BP_2,Diastolic_BP_3,Pulse_1,Pulse_2,Pulse_3,Systolic_BP_Avg,Diastolic_BP_Avg,Currently_Taking_BP_Meds
0,130378.0,135.0,131.0,132.0,98.0,96.0,94.0,82.0,79.0,82.0,133.0,96.0,1.0
1,130379.0,121.0,117.0,113.0,84.0,76.0,76.0,72.0,71.0,73.0,117.0,79.0,1.0
2,130380.0,111.0,112.0,104.0,79.0,80.0,76.0,84.0,83.0,77.0,109.0,78.0,NaN
3,130386.0,110.0,120.0,115.0,72.0,74.0,75.0,59.0,64.0,64.0,115.0,74.0,NaN
4,130387.0,143.0,136.0,145.0,76.0,74.0,78.0,80.0,80.0,77.0,141.0,76.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7796,142306.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7797,142307.0,127.0,132.0,131.0,75.0,73.0,72.0,71.0,70.0,67.0,130.0,73.0,NaN
7798,142308.0,106.0,106.0,112.0,65.0,69.0,74.0,58.0,61.0,69.0,108.0,69.0,NaN
7799,142309.0,127.0,125.0,128.0,81.0,82.0,81.0,80.0,79.0,83.0,127.0,81.0,NaN


In [55]:
# Access if the participant has hypertension or not
    # participant is considered hypertensive if either their average systolic BP is ≥ 130 mmHg or their average diastolic BP is ≥ 80 mmHg.
    # the definition below aligns with the current guidelines from organizations like the American College of Cardiology/American Heart Association (ACC/AHA) for Stage 1 Hypertension.
        # https://newsroom.heart.org/news/high-blood-pressure-redefined-for-first-time-in-14-years-130-is-the-new-high#:~:text=High%20blood%20pressure%20is%20now%20defined%20as%20readings,of%2080%20and%20higher%20for%20the%20diastolic%20measurement.
df_bp_merged['Hypertensive'] = (
    (df_bp_merged['Systolic_BP_Avg'] >= 130) | 
    (df_bp_merged['Diastolic_BP_Avg'] >= 80) |
    (df_bp_merged['Currently_Taking_BP_Meds'] == 1) # adding condition to include normal systolic or diastolic numbers while taking bp meds 
)
df_bp_merged

,Participant_ID,Systolic_BP_1,Systolic_BP_2,Systolic_BP_3,Diastolic_BP_1,Diastolic_BP_2,Diastolic_BP_3,Pulse_1,Pulse_2,Pulse_3,Systolic_BP_Avg,Diastolic_BP_Avg,Currently_Taking_BP_Meds,Hypertensive
0,130378.0,135.0,131.0,132.0,98.0,96.0,94.0,82.0,79.0,82.0,133.0,96.0,1.0,True
1,130379.0,121.0,117.0,113.0,84.0,76.0,76.0,72.0,71.0,73.0,117.0,79.0,1.0,True
2,130380.0,111.0,112.0,104.0,79.0,80.0,76.0,84.0,83.0,77.0,109.0,78.0,NaN,False
3,130386.0,110.0,120.0,115.0,72.0,74.0,75.0,59.0,64.0,64.0,115.0,74.0,NaN,False
4,130387.0,143.0,136.0,145.0,76.0,74.0,78.0,80.0,80.0,77.0,141.0,76.0,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7796,142306.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
7797,142307.0,127.0,132.0,131.0,75.0,73.0,72.0,71.0,70.0,67.0,130.0,73.0,NaN,True
7798,142308.0,106.0,106.0,112.0,65.0,69.0,74.0,58.0,61.0,69.0,108.0,69.0,NaN,False
7799,142309.0,127.0,125.0,128.0,81.0,82.0,81.0,80.0,79.0,83.0,127.0,81.0,NaN,True


In [56]:
df_bp_merged = df_bp_merged.convert_dtypes() 
df_bp_merged

,Participant_ID,Systolic_BP_1,Systolic_BP_2,Systolic_BP_3,Diastolic_BP_1,Diastolic_BP_2,Diastolic_BP_3,Pulse_1,Pulse_2,Pulse_3,Systolic_BP_Avg,Diastolic_BP_Avg,Currently_Taking_BP_Meds,Hypertensive
0,130378,135,131,132,98,96,94,82,79,82,133,96,1,True
1,130379,121,117,113,84,76,76,72,71,73,117,79,1,True
2,130380,111,112,104,79,80,76,84,83,77,109,78,<NA>,False
3,130386,110,120,115,72,74,75,59,64,64,115,74,<NA>,False
4,130387,143,136,145,76,74,78,80,80,77,141,76,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7796,142306,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False
7797,142307,127,132,131,75,73,72,71,70,67,130,73,<NA>,True
7798,142308,106,106,112,65,69,74,58,61,69,108,69,<NA>,False
7799,142309,127,125,128,81,82,81,80,79,83,127,81,<NA>,True


In [57]:
print("Combined Blood pressure examination and questionaire Data - cleaned:")
df_bp_merged.info() 

Combined Blood pressure examination and questionaire Data - cleaned:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7801 entries, 0 to 7800
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Participant_ID            7801 non-null   Int64  
 1   Systolic_BP_1             7517 non-null   Int64  
 2   Systolic_BP_2             7505 non-null   Int64  
 3   Systolic_BP_3             7480 non-null   Int64  
 4   Diastolic_BP_1            7517 non-null   Int64  
 5   Diastolic_BP_2            7505 non-null   Int64  
 6   Diastolic_BP_3            7480 non-null   Int64  
 7   Pulse_1                   7517 non-null   Int64  
 8   Pulse_2                   7505 non-null   Int64  
 9   Pulse_3                   7480 non-null   Int64  
 10  Systolic_BP_Avg           7518 non-null   Int64  
 11  Diastolic_BP_Avg          7518 non-null   Int64  
 12  Currently_Taking_BP_Meds  2333 non-null   Int64  

In [58]:
# export as csv
file_path = "../data/combined_blood_pressure_data.csv" 

try:
    df_bp_merged.to_csv(file_path, index=False, encoding='utf-8')
    print(f"DataFrame successfully saved to: {file_path}")
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")

DataFrame successfully saved to: ../data/combined_blood_pressure_data.csv


In [59]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 

In [61]:
df_bp_merged.to_sql(name = 'blood_pressure_combined_data', con=engine, schema='capstone_group_3',if_exists='replace',index=False)

801